# t0d0

* add one more column with the argmax of the q05
* Come up with a more lenient metric that will give you about 70% annotation

test

# Packages

In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import os
import spatialdata as spd

from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Data

# Locations

In [2]:
# setting up addresses

# proj_folder       = Path("/coh_labs/yunroseli/Jona/CAR-T/results/cell2location/FirstPass/") # Cohorts
# proj_folder       = Path("/home/janzules/spatial/CAR-T/data/cell2location") # Gemini - first pass files
# proj_folder     = Path("/Users/janzules/Roselab/Spatial/CAR_T/data/cell2location/")
proj_folder   = Path("/coh_labs/yunroseli/Jona/CAR-T/") # Most up to date files

out_figs          = proj_folder / "figures/manual_qc"
adata_out_folder  = proj_folder / "annotated_adata"
adata_out_folder.mkdir(exist_ok=True)
out_figs.mkdir(exist_ok=True)
# Files

zarr_file = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/Caris_Mouse_OME_output/Zarr/concatenated_tissues_FINAL"
adata_ref = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/annotating_references/predicted_cells_lvl2_epochs_200_Ncells1_decalpha_100_posterior400_bs8192.h5ad"
out_zarr  = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"

# zarr_file         = proj_folder / "data/zarr/CellCharterClusters"
# adata_ref        = proj_folder / "results/cell2location/C2L_inputs/predicted_cells_lvl2_epochs_200_Ncells1_decalpha_100.h5ad"
# adata_out         = adata_out_folder / "sp_200_epochs_defaultalpha.h5ad" # moved to the save section under annotation

## Processing data

In [3]:
# Load data
adata_vis = sc.read_h5ad(adata_ref)
zdata = spd.read_zarr(zarr_file)

/tmp/ipykernel_77857/1391359.py:3: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zdata = spd.read_zarr(zarr_file)


In [4]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

def normalize_celltype_cols(df: pd.DataFrame) -> pd.DataFrame:
    # strips everything up to and including "_sf_" so columns become just: T_cell, NK_cell, ...
    new_cols = [re.sub(r"^.*?_sf_", "", c) for c in df.columns]
    out = df.copy()
    out.columns = new_cols
    return out

means_df = normalize_celltype_cols(adata_vis.obsm["means_cell_abundance_w_sf"])
q05_df   = normalize_celltype_cols(adata_vis.obsm["q05_cell_abundance_w_sf"])
q95_df   = normalize_celltype_cols(adata_vis.obsm["q95_cell_abundance_w_sf"])

# force same set + same order
common = sorted(set(means_df.columns) & set(q05_df.columns) & set(q95_df.columns))
means_df = means_df[common]
q05_df   = q05_df[common]
q95_df   = q95_df[common]

assert (means_df.columns == q05_df.columns).all()
assert (means_df.columns == q95_df.columns).all()


# Annotating

In [5]:
def annotate_cell2location_dom(
    means_df: pd.DataFrame,
    q05_df: pd.DataFrame,
    q95_df: pd.DataFrame,
    q_cut: float = 0.05,
    dom_cut: float = 1.3,
    dom_cons_cut: float = 0.35,
    sep_cut: float = -np.inf,
    min_total_mean: float = 0.25,
    unknown_label: str = "Unknown",
):
    assert (means_df.columns == q05_df.columns).all()
    assert (means_df.columns == q95_df.columns).all()

    M = means_df.to_numpy()
    Q05 = q05_df.to_numpy()
    Q95 = q95_df.to_numpy()
    n, k = M.shape

    # winner by mean
    win_idx = M.argmax(axis=1)
    win_mean = M[np.arange(n), win_idx]
    win_q05  = Q05[np.arange(n), win_idx]

    total_mean = M.sum(axis=1)

    # runner-up by mean
    top2_idx = np.argpartition(M, -2, axis=1)[:, -2:]
    ru_idx = np.where(top2_idx[:, 0] == win_idx, top2_idx[:, 1], top2_idx[:, 0])
    ru_mean = M[np.arange(n), ru_idx]
    ru_q95  = Q95[np.arange(n), ru_idx]

    dom = win_mean / (ru_mean + 1e-12)
    dom_cons = win_q05 / (ru_q95 + 1e-12)
    sep = win_q05 - ru_q95

    ok = (
        (total_mean >= min_total_mean) &
        (win_q05 >= q_cut) &
        (dom >= dom_cut) &
        (dom_cons >= dom_cons_cut) &
        (sep >= sep_cut)
    )

    celltypes = means_df.columns.to_numpy()
    labels = np.where(ok, celltypes[win_idx], unknown_label)

    out = pd.DataFrame({
        "label": labels,
        "winner_celltype": celltypes[win_idx],
        "winner_mean": win_mean,
        "winner_q05": win_q05,
        "runnerup_celltype": celltypes[ru_idx],
        "runnerup_mean": ru_mean,
        "runnerup_q95": ru_q95,
        "dom_top1_over_top2": dom,
        "dom_conservative": dom_cons,
        "sep_q05_minus_runnerup_q95": sep,
        "total_abundance_mean": total_mean,
        "is_high_conf": ok,
    }, index=means_df.index)

    return out


In [6]:
ann_perm = annotate_cell2location_dom(
    means_df, q05_df, q95_df,
    q_cut=0.01,
    dom_cut=1.01,
    dom_cons_cut=0.105,
    sep_cut=-np.inf,
    min_total_mean=0.2,
)

ann_strict = annotate_cell2location_dom(
    means_df, q05_df, q95_df,
    q_cut=0.01,
    dom_cut=1.2,
    dom_cons_cut=0.2,
    sep_cut=-np.inf,
    min_total_mean=0.2,
)

# argmax: always assigns, no quality filter
c2l_argmax = means_df.idxmax(axis=1)

for name, labels in [("argmax", c2l_argmax), ("permissive", ann_perm["label"]), ("strict", ann_strict["label"])]:
    labeled = (labels != "Unknown").sum()
    total = len(labels)
    print(f"\n{'='*40}")
    print(f"{name}: {labeled:,} / {total:,} ({labeled/total*100:.1f}%) annotated")
    print(labels.value_counts().to_string())



argmax: 2,088,557 / 2,088,557 (100.0%) annotated
Erythrocyte          556344
N1_like_Neu          452153
Cancer_cell          370928
N2_like_Neu          346361
M2_like_Mac           62447
Classical_Mono        61994
B                     51730
CD8_T                 49378
Fibroblast            25623
NK                    24190
M1_like_Mac           20504
Endothelial           18229
CD4_T                 10808
cDC                    9416
NKT                    9398
Treg                   5904
Nonclassical_Mono      5880
pDC                    5477
Intermediate_Mac       1793

permissive: 1,231,898 / 2,088,557 (59.0%) annotated
label
Unknown              856659
Cancer_cell          363019
Erythrocyte          210019
N2_like_Neu          193240
N1_like_Neu          134656
M2_like_Mac           60720
Classical_Mono        58279
CD8_T                 45824
B                     40422
Fibroblast            24867
NK                    22924
M1_like_Mac           19170
Endothelial           1

In [7]:
# 1) Reindex to match zarr order
z_adata = zdata.tables["segmentation_counts"]

# 2) Three annotation labels — runner-up is threshold-agnostic (same for all)
z_adata.obs["c2l_argmax"]      = c2l_argmax.reindex(z_adata.obs_names).values
z_adata.obs["c2l_permissive"]  = ann_perm["label"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_strict"]      = ann_strict["label"].reindex(z_adata.obs_names).values
z_adata.obs["c2l_runnerup"]    = ann_perm["runnerup_celltype"].reindex(z_adata.obs_names).values

# 3) Full abundance matrices → .obsm
z_adata.obsm["c2l_means"] = means_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q05"]   = q05_df.reindex(z_adata.obs_names)
z_adata.obsm["c2l_q95"]   = q95_df.reindex(z_adata.obs_names)


In [8]:
import numpy as np
import pandas as pd

assert z_adata.n_obs == adata_vis.n_obs, "n_obs mismatch"
assert set(z_adata.obs_names) == set(adata_vis.obs_names), "obs_names sets differ"
print(f"[OK] Row alignment: {z_adata.n_obs:,} cells")

# --- annotation columns ---
for col, src in [
    ("c2l_argmax",     c2l_argmax),
    ("c2l_permissive", ann_perm["label"]),
    ("c2l_strict",     ann_strict["label"]),
    ("c2l_runnerup",   ann_perm["runnerup_celltype"]),
]:
    assert col in z_adata.obs.columns, f"{col} missing from z_adata.obs"
    left  = z_adata.obs[col].astype(str).values
    right = src.reindex(z_adata.obs_names).astype(str).values
    n_mismatch = (left != right).sum()
    n_nan = pd.isna(z_adata.obs[col]).sum()
    assert n_mismatch == 0, f"{col}: {n_mismatch} mismatched values"
    print(f"[OK] {col}: 0 mismatches, {n_nan} NaN")

# --- obsm matrices ---
for key, src_df in [("c2l_means", means_df), ("c2l_q05", q05_df), ("c2l_q95", q95_df)]:
    assert key in z_adata.obsm, f"{key} missing from z_adata.obsm"
    dst = z_adata.obsm[key]
    assert dst.shape == src_df.shape, f"{key}: shape {dst.shape} vs {src_df.shape}"
    dst_arr = dst.loc[z_adata.obs_names].to_numpy() if isinstance(dst, pd.DataFrame) else np.asarray(dst)
    src_arr = src_df.reindex(z_adata.obs_names).to_numpy()
    max_diff = np.nanmax(np.abs(dst_arr - src_arr))
    assert max_diff < 1e-10, f"{key}: max abs diff = {max_diff}"
    print(f"[OK] {key}: shape={dst.shape}, max|diff|={max_diff:.2e}")

# --- label distribution summary ---
print()
for col in ["c2l_argmax", "c2l_permissive", "c2l_strict"]:
    unknown_frac = (z_adata.obs[col] == "Unknown").mean()
    print(f"{col}: {(1-unknown_frac)*100:.1f}% annotated")


[OK] Row alignment: 2,088,557 cells
[OK] c2l_argmax: 0 mismatches, 0 NaN
[OK] c2l_permissive: 0 mismatches, 0 NaN
[OK] c2l_strict: 0 mismatches, 0 NaN
[OK] c2l_runnerup: 0 mismatches, 0 NaN
[OK] c2l_means: shape=(2088557, 19), max|diff|=0.00e+00
[OK] c2l_q05: shape=(2088557, 19), max|diff|=0.00e+00
[OK] c2l_q95: shape=(2088557, 19), max|diff|=0.00e+00

c2l_argmax: 100.0% annotated
c2l_permissive: 59.0% annotated
c2l_strict: 31.4% annotated


## Consolidated annotation

Starts from `c2l_permissive`. All labeled cells are carried over. For `Unknown` cells, applies conditional grouping based on `c2l_argmax` (best-by-mean) and `c2l_runnerup` (second-best-by-mean):

| argmax / runnerup (bidirectional) | → label |
|---|---|
| N1_like_Neu ↔ N2_like_Neu | Neutrophil |
| any two of {M1_like_Mac, M2_like_Mac, Intermediate_Mac} | Macrophage |
| Classical_Mono ↔ Nonclassical_Mono | Monocyte |
| CD8_T ↔ CD4_T | Tcell |
| CD4_T ↔ Treg | Tcell |
| cDC ↔ pDC | DC |

In [9]:
perm     = z_adata.obs["c2l_permissive"].astype(str)
argmax   = z_adata.obs["c2l_argmax"].astype(str)
runnerup = z_adata.obs["c2l_runnerup"].astype(str)

unknown_mask = perm == "Unknown"
consolidated = perm.copy()

def _pair(a, b, x, y):
    """True where {a,b} == {x,y} (bidirectional)."""
    return ((a == x) & (b == y)) | ((a == y) & (b == x))

# --- Conditional grouping for Unknown cells ---
MAC_TYPES = {"M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"}

rules = [
    # (mask, new_label)
    (unknown_mask & _pair(argmax, runnerup, "N1_like_Neu",     "N2_like_Neu"),      "Neutrophil"),
    (unknown_mask & argmax.isin(MAC_TYPES) & runnerup.isin(MAC_TYPES),              "Macrophage"),
    (unknown_mask & _pair(argmax, runnerup, "Classical_Mono",  "Nonclassical_Mono"),"Monocyte"),
    (unknown_mask & _pair(argmax, runnerup, "CD8_T",           "CD4_T"),            "Tcell"),
    (unknown_mask & _pair(argmax, runnerup, "CD4_T",           "Treg"),             "Tcell"),
    (unknown_mask & _pair(argmax, runnerup, "cDC",             "pDC"),              "DC"),
]

for mask, label in rules:
    n = mask.sum()
    if n:
        consolidated[mask] = label
        print(f"  {label}: +{n:,} from Unknown")

z_adata.obs["c2l_consolidated"] = consolidated.values

# Summary
print("\nc2l_consolidated distribution:")
print(z_adata.obs["c2l_consolidated"].value_counts().to_string())
still_unknown = (consolidated == "Unknown").sum()
print(f"\nRemaining Unknown: {still_unknown:,} / {len(consolidated):,} ({still_unknown/len(consolidated)*100:.1f}%)")

  Neutrophil: +184,827 from Unknown
  Macrophage: +145 from Unknown
  Monocyte: +113 from Unknown
  Tcell: +105 from Unknown
  Tcell: +15 from Unknown
  DC: +20 from Unknown

c2l_consolidated distribution:
c2l_consolidated
Unknown              671434
Cancer_cell          363019
Erythrocyte          210019
N2_like_Neu          193240
Neutrophil           184827
N1_like_Neu          134656
M2_like_Mac           60720
Classical_Mono        58279
CD8_T                 45824
B                     40422
Fibroblast            24867
NK                    22924
M1_like_Mac           19170
Endothelial           17702
CD4_T                 10085
cDC                    8471
NKT                    7110
Treg                   5155
pDC                    4558
Nonclassical_Mono      4002
Intermediate_Mac       1675
Macrophage              145
Tcell                   120
Monocyte                113
DC                       20

Remaining Unknown: 671,434 / 2,088,557 (32.1%)


In [9]:
# Drop any stale c2l columns from a previous run before writing
stale = ["c2l_winner", "c2l_is_high_conf", "c2l_label_perm", "c2l_label_strict", "c2l_runnerup_celltype"]
to_drop = [c for c in stale if c in z_adata.obs.columns]
if to_drop:
    z_adata.obs.drop(columns=to_drop, inplace=True)
    print("Dropped:", to_drop)

print("obs columns with c2l:", [c for c in z_adata.obs.columns if c.startswith("c2l_")])


obs columns with c2l: ['c2l_argmax', 'c2l_permissive', 'c2l_strict', 'c2l_runnerup']


# Further Processing

## Rename and normalize elements

Rename ambiguous/inconsistent tissue names before any downstream steps:
- `RTCyPSCA_1_2_1` → `RTCyPSCA_1_2` (canonical; the `_1` suffix was a processing artifact)
- `RTCyPSCA_1_2_2` → `RTCyPSCA_1_5` (kept in main as "replicate 5"; all other replicates are 1–4, so this stands out for downstream filtering)
- `NoTX` → `NoTx` (capitalization normalization)

Applied to images, shapes, `obs["tissue"]`, `obs["region"]`, and `uns["spatialdata_attrs"]["region"]`.

In [10]:
RENAMES = {
    "RTCyPSCA_1_2_1": "RTCyPSCA_1_2",  # canonical; _1 suffix was a processing artifact
    "RTCyPSCA_1_2_2": "RTCyPSCA_1_5",  # keep in main as "replicate 5" (all others are 1–4)
    "NoTX": "NoTx",                      # normalize capitalization
}

def _rename(name, mapping):
    for old, new in mapping.items():
        if name == old:
            return new
        if name.startswith(old + "_"):
            return new + name[len(old):]
    return name

def _rename_obs(adata, cols, renames):
    for col in cols:
        if col not in adata.obs.columns:
            continue
        orig = adata.obs[col]
        was_cat = isinstance(orig.dtype, pd.CategoricalDtype)
        old_vals = orig.astype(str)
        new_vals = old_vals.map(lambda v: _rename(v, renames))
        n_changed = (new_vals != old_vals).sum()
        if n_changed:
            adata.obs[col] = pd.Categorical(new_vals.values) if was_cat else new_vals.values
            print(f"  {col}: {n_changed:,} values renamed  (categorical={was_cat})")

z_adata = zdata.tables["segmentation_counts"]

# --- 1) Rename image/shape element keys ---
print("Renaming images:")
for old_name in list(zdata.images.keys()):
    new_name = _rename(old_name, RENAMES)
    if new_name != old_name:
        zdata.images[new_name] = zdata.images.pop(old_name)
        print(f"  {old_name} -> {new_name}")

print("Renaming shapes:")
for old_name in list(zdata.shapes.keys()):
    new_name = _rename(old_name, RENAMES)
    if new_name != old_name:
        zdata.shapes[new_name] = zdata.shapes.pop(old_name)
        print(f"  {old_name} -> {new_name}")

# --- 2) Build renamed table (copy + rename obs + rename uns) before assigning ---
renamed_table = z_adata.copy()
print("\nRenaming obs:")
_rename_obs(renamed_table, ["tissue", "region"], RENAMES)

if "spatialdata_attrs" in renamed_table.uns:
    attrs = dict(renamed_table.uns["spatialdata_attrs"])
    regions = attrs.get("region", [])
    if isinstance(regions, str):
        regions = [regions]
    attrs["region"] = [_rename(r, RENAMES) for r in regions]
    renamed_table.uns["spatialdata_attrs"] = attrs
    print(f"  uns['spatialdata_attrs']['region']: updated")

# --- 3) Assign back (obs["region"] and uns["region"] now consistent) ---
zdata.tables["segmentation_counts"] = renamed_table
z_adata = zdata.tables["segmentation_counts"]

print(f"\n[OK] zdata: {len(zdata.images)} images, {len(zdata.shapes)} shapes, {z_adata.n_obs:,} cells")
print(f"Unique tissues: {sorted(z_adata.obs['tissue'].astype(str).unique())}")

Renaming images:
  RTCyPSCA_1_2_2_hires_tissue_image -> RTCyPSCA_1_5_hires_tissue_image
  RTCyPSCA_1_2_1_hires_tissue_image -> RTCyPSCA_1_2_hires_tissue_image
  NoTX_2_4_hires_tissue_image -> NoTx_2_4_hires_tissue_image
Renaming shapes:
  RTCyPSCA_1_2_1_cell_boundaries -> RTCyPSCA_1_2_cell_boundaries
  NoTX_2_4_roi -> NoTx_2_4_roi
  RTCyPSCA_1_2_1_roi -> RTCyPSCA_1_2_roi
  RTCyPSCA_1_2_2_cell_boundaries -> RTCyPSCA_1_5_cell_boundaries
  NoTX_2_4_cell_boundaries -> NoTx_2_4_cell_boundaries
  RTCyPSCA_1_2_2_roi -> RTCyPSCA_1_5_roi

Renaming obs:
  tissue: 151,102 values renamed  (categorical=False)
  region: 151,102 values renamed  (categorical=True)
  uns['spatialdata_attrs']['region']: updated

[OK] zdata: 32 images, 64 shapes, 2,088,557 cells
Unique tissues: ['CyPSCA_1_1', 'CyPSCA_1_2', 'CyPSCA_1_3', 'CyPSCA_1_4', 'CyPSCA_2_1', 'CyPSCA_2_2', 'CyPSCA_2_3', 'CyPSCA_2_4', 'CyT72_1_2', 'CyT72_1_4', 'CyT72_2_3', 'CyT72_2_4', 'NoTx_1_1', 'NoTx_1_2', 'NoTx_1_3', 'NoTx_1_4', 'NoTx_2_1', 'NoTx

In [11]:
# --- Sanity checks: rename + normalize ---
STALE_TOKENS = ["NoTX", "RTCyPSCA_1_2_1", "RTCyPSCA_1_2_2"]
EXPECTED_NEW = ["NoTx_2_4", "RTCyPSCA_1_2", "RTCyPSCA_1_5"]

failures = []

def _check(cond, msg):
    status = "OK " if cond else "FAIL"
    print(f"  [{status}] {msg}")
    if not cond:
        failures.append(msg)

# --- 1) Total cell count unchanged ---
print("Counts:")
_check(z_adata.n_obs == 2_088_557, f"n_obs = 2,088,557 (got {z_adata.n_obs:,})")

# --- 2) No stale tokens anywhere ---
print("\nStale-token scan:")
for tok in STALE_TOKENS:
    in_tissue = z_adata.obs["tissue"].astype(str).str.contains(tok, regex=False).any()
    in_region = z_adata.obs["region"].astype(str).str.contains(tok, regex=False).any()
    in_images = any(tok in k for k in zdata.images)
    in_shapes = any(tok in k for k in zdata.shapes)
    in_uns    = any(tok in r for r in z_adata.uns.get("spatialdata_attrs", {}).get("region", []))
    _check(not (in_tissue or in_region or in_images or in_shapes or in_uns),
           f"'{tok}' absent everywhere "
           f"(tissue={in_tissue}, region={in_region}, images={in_images}, shapes={in_shapes}, uns={in_uns})")

# --- 3) Expected renamed names present ---
print("\nExpected renamed names:")
for new_name in EXPECTED_NEW:
    _check(new_name in z_adata.obs["tissue"].astype(str).unique(),
           f"'{new_name}' in obs['tissue']")
    _check(any(k.startswith(new_name + "_") for k in zdata.shapes),
           f"shape starting with '{new_name}_' exists")
    _check(any(k.startswith(new_name + "_") for k in zdata.images),
           f"image starting with '{new_name}_' exists")

# --- 4) obs['region'] is categorical and matches uns ---
print("\nRegion dtype + uns consistency:")
_check(isinstance(z_adata.obs["region"].dtype, pd.CategoricalDtype),
       f"obs['region'] is Categorical (got {z_adata.obs['region'].dtype})")
uns_regions = set(z_adata.uns["spatialdata_attrs"]["region"])
obs_regions = set(z_adata.obs["region"].astype(str).unique())
_check(uns_regions == obs_regions,
       f"uns['region'] == obs['region'] unique "
       f"(only-in-uns={uns_regions - obs_regions}, only-in-obs={obs_regions - uns_regions})")

# --- 5) Every region in uns has a matching shape ---
print("\nuns['region'] vs zdata.shapes:")
missing_shapes = [r for r in uns_regions if r not in zdata.shapes]
_check(not missing_shapes, f"every uns region has a shape (missing: {missing_shapes})")

print("\n" + ("=" * 50))
if failures:
    print(f"[!] {len(failures)} CHECK(S) FAILED:")
    for f in failures:
        print(f"    - {f}")
else:
    print("[OK] All sanity checks passed.")

Counts:
  [OK ] n_obs = 2,088,557 (got 2,088,557)

Stale-token scan:
  [OK ] 'NoTX' absent everywhere (tissue=False, region=False, images=False, shapes=False, uns=False)
  [OK ] 'RTCyPSCA_1_2_1' absent everywhere (tissue=False, region=False, images=False, shapes=False, uns=False)
  [OK ] 'RTCyPSCA_1_2_2' absent everywhere (tissue=False, region=False, images=False, shapes=False, uns=False)

Expected renamed names:
  [OK ] 'NoTx_2_4' in obs['tissue']
  [OK ] shape starting with 'NoTx_2_4_' exists
  [OK ] image starting with 'NoTx_2_4_' exists
  [OK ] 'RTCyPSCA_1_2' in obs['tissue']
  [OK ] shape starting with 'RTCyPSCA_1_2_' exists
  [OK ] image starting with 'RTCyPSCA_1_2_' exists
  [OK ] 'RTCyPSCA_1_5' in obs['tissue']
  [OK ] shape starting with 'RTCyPSCA_1_5_' exists
  [OK ] image starting with 'RTCyPSCA_1_5_' exists

Region dtype + uns consistency:
  [OK ] obs['region'] is Categorical (got category)
  [OK ] uns['region'] == obs['region'] unique (only-in-uns=set(), only-in-obs=set())

## Centroid

In [12]:
import pandas as pd
import numpy as np

# Compute centroids from cell boundary geometries
centroid_frames = []

for shp_name, gdf in zdata.shapes.items():
    if not shp_name.endswith("_cell_boundaries"):
        continue

    centroids = gdf.geometry.centroid
    df = pd.DataFrame({
        "cell_id": gdf.index.astype(str),
        "x": centroids.x.values,
        "y": centroids.y.values,
    }).set_index("cell_id")

    print(f"{shp_name}: {len(df)} centroids")
    centroid_frames.append(df)

centroids_all = pd.concat(centroid_frames, axis=0)
print("Total centroids:", len(centroids_all))

if centroids_all.index.duplicated().any():
    n_dup = centroids_all.index.duplicated().sum()
    print(f"WARNING: {n_dup} duplicate cell_ids in shapes — keeping first occurrence.")
    centroids_all = centroids_all[~centroids_all.index.duplicated(keep="first")]

# Spot-check: verify ID format matches between shapes and z_adata before joining
print("\nShape GDF index sample: ", centroids_all.index[:5].tolist())
print("z_adata cell_id sample: ", z_adata.obs["cell_id"].astype(str).iloc[:5].tolist())

# Align with z_adata using cell_id and update obsm["spatial"]
cell_ids = z_adata.obs["cell_id"].astype(str)

missing = set(cell_ids) - set(centroids_all.index)
if missing:
    print(f"WARNING: {len(missing)} cell_ids in z_adata have no matching shape centroid.")
    print("Example missing IDs:", list(sorted(missing))[:10])

centroids_ordered = centroids_all.reindex(cell_ids)

if centroids_ordered.isna().any().any():
    raise ValueError("Some centroids are NaN — resolve missing cell_ids before proceeding.")

z_adata.obsm["spatial"] = centroids_ordered[["x", "y"]].to_numpy(dtype=float)
print(f"[OK] obsm['spatial'] updated: shape={z_adata.obsm['spatial'].shape}")


CyPSCA_1_4_cell_boundaries: 27025 centroids
NoTx_1_2_cell_boundaries: 23792 centroids
RTCyT72_2_1_cell_boundaries: 76611 centroids
CyPSCA_1_1_cell_boundaries: 108895 centroids
CyT72_1_4_cell_boundaries: 49785 centroids
RTCyT72_1_2_cell_boundaries: 71091 centroids
NoTx_1_4_cell_boundaries: 34688 centroids
RTCyT72_2_2_cell_boundaries: 71223 centroids
CyPSCA_2_4_cell_boundaries: 54121 centroids
NoTx_1_3_cell_boundaries: 59018 centroids
RTCyPSCA_2_3_cell_boundaries: 112486 centroids
NoTx_2_1_cell_boundaries: 55827 centroids
CyPSCA_1_3_cell_boundaries: 33085 centroids
RTCyPSCA_2_4_cell_boundaries: 65157 centroids
RTCyPSCA_1_4_cell_boundaries: 69639 centroids
NoTx_1_1_cell_boundaries: 44929 centroids
CyT72_2_4_cell_boundaries: 107546 centroids
RTCyT72_1_1_cell_boundaries: 32257 centroids
RTCyT72_1_4_cell_boundaries: 65460 centroids
CyPSCA_2_2_cell_boundaries: 61384 centroids
CyPSCA_2_1_cell_boundaries: 86108 centroids
NoTx_2_2_cell_boundaries: 131488 centroids
RTCyPSCA_1_3_cell_boundaries: 5

## OBS

In [13]:
# Strip TMA suffix: keep only the F##### prefix
# e.g., F07839_1n2_OME -> F07839, F08543_OME -> F08543
before = sorted(z_adata.obs["TMA"].astype(str).unique())
z_adata.obs["TMA"] = z_adata.obs["TMA"].astype(str).str.split("_", n=1).str[0]
after = sorted(z_adata.obs["TMA"].unique())

print(f"TMA before: {before}")
print(f"TMA after:  {after}")


TMA before: ['F07839_1n2_OME', 'F07840_123_OME', 'F08542_1n3_OME', 'F08543_OME', 'F08635_OME', 'F08636_OME', 'F08637_OME', 'F08638_OME']
TMA after:  ['F07839', 'F07840', 'F08542', 'F08543', 'F08635', 'F08636', 'F08637', 'F08638']


In [14]:
# Parse tissue = [condition]_[tumor_loc]_[replicate_num]
# treatment    = [condition]_Tu[tumor_loc]
tissue_str = z_adata.obs["tissue"].astype(str)
parts = tissue_str.str.rsplit("_", n=2, expand=True)
parts.columns = ["condition", "tumor_loc", "replicate_num"]

z_adata.obs["condition"]     = parts["condition"].values
z_adata.obs["tumor_loc"]     = parts["tumor_loc"].values
z_adata.obs["replicate_num"] = parts["replicate_num"].values
z_adata.obs["treatment"]     = (parts["condition"].astype(str) + "_Tu" + parts["tumor_loc"].astype(str)).values

# Verify
print("Sample parses (one row per unique tissue):")
print(
    z_adata.obs[["tissue", "condition", "tumor_loc", "replicate_num", "treatment"]]
    .drop_duplicates()
    .sort_values("tissue")
    .to_string(index=False)
)
print()
print(f"Unique conditions:    {sorted(z_adata.obs['condition'].unique())}")
print(f"Unique tumor_loc:     {sorted(z_adata.obs['tumor_loc'].unique())}")
print(f"Unique replicate_num: {sorted(z_adata.obs['replicate_num'].unique())}")
print(f"Unique treatments:    {sorted(z_adata.obs['treatment'].unique())}")


Sample parses (one row per unique tissue):
      tissue condition tumor_loc replicate_num    treatment
  CyPSCA_1_1    CyPSCA         1             1   CyPSCA_Tu1
  CyPSCA_1_2    CyPSCA         1             2   CyPSCA_Tu1
  CyPSCA_1_3    CyPSCA         1             3   CyPSCA_Tu1
  CyPSCA_1_4    CyPSCA         1             4   CyPSCA_Tu1
  CyPSCA_2_1    CyPSCA         2             1   CyPSCA_Tu2
  CyPSCA_2_2    CyPSCA         2             2   CyPSCA_Tu2
  CyPSCA_2_3    CyPSCA         2             3   CyPSCA_Tu2
  CyPSCA_2_4    CyPSCA         2             4   CyPSCA_Tu2
   CyT72_1_2     CyT72         1             2    CyT72_Tu1
   CyT72_1_4     CyT72         1             4    CyT72_Tu1
   CyT72_2_3     CyT72         2             3    CyT72_Tu2
   CyT72_2_4     CyT72         2             4    CyT72_Tu2
    NoTx_1_1      NoTx         1             1     NoTx_Tu1
    NoTx_1_2      NoTx         1             2     NoTx_Tu1
    NoTx_1_3      NoTx         1             3     NoTx_T

## Normalization

In [15]:
# Preserve raw counts before normalization
z_adata.layers["counts"] = z_adata.X.copy()

sc.pp.normalize_total(z_adata, target_sum=1e4, inplace=True)
sc.pp.log1p(z_adata)

z_adata.uns["normalization"] = {
    "method": "scanpy.pp.normalize_total + log1p",
    "target_sum": 1e4,
    "raw_counts_layer": "counts",
}

print("[OK] Normalized X  →  log1p(CP10k)")
print(f"     Raw counts preserved in layers['counts']")
print(f"     X shape: {z_adata.X.shape}")
print(f"     Layers:  {list(z_adata.layers.keys())}")

[OK] Normalized X  →  log1p(CP10k)
     Raw counts preserved in layers['counts']
     X shape: (2088557, 19059)
     Layers:  ['counts']


## Saving

In [17]:
# 1) Push the updated table back into the SpatialData object
zdata.tables["segmentation_counts"] = z_adata


In [18]:

# 2) Write to out_zarr (defined in the locations cell)
zdata.write(out_zarr, overwrite=True)
print("Wrote to:", out_zarr)

# 3) Quick verify — reload and spot-check
import spatialdata as spd
zdata_check = spd.read_zarr(out_zarr)
t = zdata_check.tables["segmentation_counts"]


Wrote to: /coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400


In [19]:

print("Reloaded shape:", t.shape)
print("obs columns with c2l:", [c for c in t.obs.columns if c.startswith("c2l_")])
print("obsm keys:", list(t.obsm.keys()))
print(t.obs["c2l_permissive"].value_counts().head(5))

Reloaded shape: (2088557, 19059)
obs columns with c2l: ['c2l_argmax', 'c2l_permissive', 'c2l_strict', 'c2l_runnerup', 'c2l_consolidated']
obsm keys: ['c2l_q95', 'spatial', 'c2l_q05', 'c2l_means']
c2l_permissive
Unknown        856659
Cancer_cell    363019
Erythrocyte    210019
N2_like_Neu    193240
N1_like_Neu    134656
Name: count, dtype: int64
